<a href="https://colab.research.google.com/github/chandupatilyewale/CNN_CIFAR10/blob/main/CNN_CIFER10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
import torch
import torch.nn as nn
import torch.optim as optim

In [44]:
import torchvision
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
from torchvision.transforms import transforms

In [45]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),(0.5, 0.5, 0.5))
])

In [46]:
trainset = CIFAR10(root = "./data",train = True, download = True, transform = transform)
testset = CIFAR10(root = "./data", train = False, download = True, transform = transform)

In [47]:
trainloader = DataLoader(trainset, batch_size = 64, shuffle = True)
testloader = DataLoader(testset, batch_size = 64)

Build a CNN

In [52]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),  # kernel size = 2 and stride = 2

            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64, 128, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(4 * 4 * 128, 256),
            nn.ReLU(),
            nn.Linear(256, 10),
        )
    def forword(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1)  # flattening
        x = self.fc_layers(x)
        return x

In [53]:
model = CNN()

In [54]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

Training the CNN

In [56]:
epochs = 10
for epoch in range(epochs):
    epoch_training_loss = 0
    for images, labels in trainloader:
        optimizer.zero_grad()
        output = model.forword(images)
        loss = criterion(output,labels)
        loss.backward()
        optimizer.step()
        epoch_training_loss += loss.item()
    print(f"epoch = {epoch + 1}/{epochs} & loss = {epoch_training_loss / len(trainloader)}")

epoch = 1/10 & loss = 1.3796020307199424
epoch = 2/10 & loss = 0.9351125195660555
epoch = 3/10 & loss = 0.737987118875584
epoch = 4/10 & loss = 0.6083435766837176
epoch = 5/10 & loss = 0.5033087167898407
epoch = 6/10 & loss = 0.41195117672690956
epoch = 7/10 & loss = 0.32547907154921374
epoch = 8/10 & loss = 0.256310095224539
epoch = 9/10 & loss = 0.19537979009492165
epoch = 10/10 & loss = 0.15400595635251926


Evaluate our CNN

In [57]:
correct_labels = 0
total_labels = 0
model.eval()
with torch.no_grad():
    for images, labels in testloader:
        output = model.forword(images)
        _, predicted = torch.max(output,1)
        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

    print(f"Accuracy = {correct_labels / total_labels * 100}")


Accuracy = 76.03999999999999
